# Lesson 02 - Истраживање Microsoft Agent Framework-а

**Microsoft Agent Framework (MAF)** је унификован оквир за израду AI агената. Обезбеђује чисту, састављиву архитектуру са четири основне градивне блокове:

- **Клијент** – повезује се на крајњу тачку AI модела и обрађује комуникацију
- **Агент** – обухвата клијента инструкцијама и дефиницијама алата
- **Алатке** – проширују могућности агента прилагођеним функцијама које модел може позивати
- **Сесија** – одржава историју разговора за интеракције са више корака

У овој лекцији, направићемо **агента за резервацију путовања** који проверава доступност дестинације користећи ове концепте.


## Подешавање


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Разумевање архитектуре Agent Framework-а

Microsoft Agent Framework прати слојевиту архитектуру:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Клијент** – `AzureAIProjectAgentProvider` се повезује на Azure OpenAI распоређивање. Он обрађује аутентификацију, форматирање захтева и парсирање одговора.
2. **Agent** – Креиран из клијента путем `provider.create_agent()`, агент комбинује приступ моделу са упутствима (системски промпт) и алатима.
3. **Алатке** – Python функције украшене са `@tool` које агент може позвати да изврши радње или преузме податке.
4. **Сесија** – Објекат `AgentSession` (направљен путем `agent.create_session()`) који чува историју конверзације, омогућавајући вишекратну комуникацију где агент памти претходни контекст.

Хајде да изграђемо сваки слој корак по корак.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Додавање алата са @tool декоратором

Алатке омогућавају агентима да изврше акције осим генерисања текста. Декоратор `@tool` претвара обичну Питхон функцију у нешто што агент може да позове.

Кључне тачке:
- Користите `Annotated[type, "description"]` да модел разуме сваки параметар.
- Докстринг постаје опис алатке који модел види.
- `approval_mode="never_require"` значи да се алатка аутоматски покреће без корисничког одобрења.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Креирање агента са алаткама

Сада комбинујемо клијента, упутства и алатке у агента. `instructions` делују као системски подсетник — они дефинишу личност и понашање агента.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Вишеокретни разговори са сесијама

`AgentSession` (направљен преко `agent.create_session()`) прате свакада поруке у разговору. Прослеђујући исту сесију сваком позиву `agent.run()`, агент има приступ целој историји разговора и може се позвати на претходне поруке.

Прослеђујемо `tools=[check_destination_availability]` тако да агент може да позове нашу проверу доступности током сваког корака.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Резиме

У овој лекцији сте истражили четири стуба Microsoft Agent Framework-а:

| Концепт | Оно што сте научили |
|---------|---------------------|
| **Клијент** | `AzureAIProjectAgentProvider` се повезује са Azure OpenAI помоћу аутентификације засноване на акредитивима |
| **Агент** | `provider.create_agent()` спаја везу са моделом са упутствима и именом |
| **Алатке** | Декоратор `@tool` излаже Python функције које агент може да позива |
| **Сесија** | `agent.create_session()` одржава историју разговора кроз више корака |

Ови грађевински блокови се комбинују да би створили агенте који могу да воде природне разговоре, позивају спољне функције и одржавају контекст — основа за напредније агенцијске обрасце у каснијим лекцијама.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Одрицање од одговорности**:
Овај документ је преведен коришћењем AI сервиса за превођење [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте на уму да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитетним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било какве неспоразуме или непредвиђена тумачења проистекла из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
